# CNN on CIFAR-10 — Real Image Classification

CIFAR-10 is a dataset of 60,000 real-world color photos across 10 categories:
✈️ airplane, 🚗 car, 🐦 bird, 🐱 cat, 🦌 deer, 🐶 dog, 🐸 frog, 🐴 horse, 🚢 ship, 🚚 truck

**What's different from MNIST:**
- Color images (3 RGB channels instead of 1)
- Real-world messiness — varying angles, lighting, backgrounds
- Much harder — we'll aim for ~75% accuracy with a basic CNN

**Architecture (deeper than MNIST CNN):**
```
Input (3×32×32)
  → Conv(32) → BN → ReLU → Conv(32) → BN → ReLU → MaxPool → Dropout
  → Conv(64) → BN → ReLU → Conv(64) → BN → ReLU → MaxPool → Dropout
  → Conv(128) → BN → ReLU → MaxPool → Dropout
  → Flatten → Linear(256) → ReLU → Dropout → Linear(10)
```
**New concept: Batch Normalization (BN)** — normalizes layer outputs during training, making it faster and more stable.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Use MPS (Apple Silicon) if available, else CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print('Using device:', device)

## 1. Load Data

We apply **data augmentation** during training — randomly flipping and cropping images so the model sees more variety and generalizes better.

In [ ]:
CLASSES = ['airplane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

# Training: augment with random flips and crops
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # normalize each RGB channel
])

# Test: no augmentation, just normalize
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_data = datasets.CIFAR10(root='data', train=True,  download=True, transform=train_transform)
test_data  = datasets.CIFAR10(root='data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_data,  batch_size=128, shuffle=False, num_workers=0)

print(f'Train: {len(train_data)} | Test: {len(test_data)}')

## 2. Visualize Sample Images

In [ ]:
images, labels = next(iter(train_loader))

# Unnormalize for display
def unnormalize(img):
    return img * 0.5 + 0.5

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = unnormalize(images[i]).permute(1, 2, 0).numpy()  # C×H×W → H×W×C
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(CLASSES[labels[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample CIFAR-10 Images')
plt.tight_layout()
plt.show()

## 3. Define the CNN

Deeper than the MNIST CNN — we need more capacity to handle complex real-world images.

In [ ]:
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3×32×32 → 32×16×16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 32→16
            nn.Dropout(0.2),

            # Block 2: 32×16×16 → 64×8×8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 16→8
            nn.Dropout(0.3),

            # Block 3: 64×8×8 → 128×4×4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 8→4
            nn.Dropout(0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),         # 128×4×4 = 2048
            nn.Linear(128*4*4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CIFAR_CNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## 4. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # halve LR every 10 epochs

EPOCHS = 20
train_losses = []
test_accuracies = []

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    # --- Evaluate ---
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            _, predicted = torch.max(model(images), 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    test_accuracies.append(acc)

    scheduler.step()
    print(f'Epoch {epoch+1:2d}/{EPOCHS}  |  Loss: {avg_loss:.4f}  |  Test Acc: {acc:.2f}%')

print(f'\nBest accuracy: {max(test_accuracies):.2f}%')

## 5. Plot Loss & Accuracy

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, EPOCHS+1), train_losses, marker='o', color='steelblue')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(range(1, EPOCHS+1), test_accuracies, marker='o', color='darkorange')
ax2.set_title('Test Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True)

plt.tight_layout()
plt.show()

## 6. Per-Class Accuracy

Which categories is the model good at? Which does it confuse?

In [ ]:
model.eval()
class_correct = [0] * 10
class_total   = [0] * 10

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = torch.max(model(images), 1)
        for label, pred in zip(labels, predicted):
            class_total[label] += 1
            if label == pred:
                class_correct[label] += 1

print('Per-class accuracy:')
for i, cls in enumerate(CLASSES):
    acc = 100 * class_correct[i] / class_total[i]
    bar = '█' * int(acc // 5)
    print(f'  {cls:10s}  {acc:5.1f}%  {bar}')

## 7. Show Predictions

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images_dev = images.to(device)

with torch.no_grad():
    _, predicted = torch.max(model(images_dev), 1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = unnormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(np.clip(img, 0, 1))
    p, t = predicted[i].item(), labels[i].item()
    color = 'green' if p == t else 'red'
    ax.set_title(f'{CLASSES[p]}', color=color, fontsize=7)
    ax.axis('off')
plt.suptitle('Predictions — green=correct, red=wrong')
plt.tight_layout()
plt.show()

## Next Steps

- **Transfer Learning** — use a pretrained ResNet/VGG and fine-tune it on CIFAR-10 → jumps to 90%+
- **More augmentation** — random rotation, color jitter, cutout
- **Larger model** — add more conv blocks or use ResNet-style skip connections